In [1]:
from services.simulation_service import simulate_circuit
from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import time

In [2]:
def dict_builder(nq, clb, initial_state, algorithm, runner_mode, noise_model, shots):
    initial_instructions = []
    for c, i in zip(initial_state, range(nq)):
        if c == '1':
            initial_instructions.append({"name": "X", "qubits": [i]})
    
    if runner_mode == 'shot': algorithm.append({"name": "MEASURE_ALL"})

    instructions = initial_instructions + algorithm
    
    return {
        "name": "Testing",
        "num_qubits": nq,
        "num_clbits": clb,
        "instructions": instructions,
        "runner_mode": runner_mode,
        "noise_model": noise_model,
        "shots": shots
    }

In [3]:
# Shor's Algorithm Test (N=15, a=7)
# We need to factor 15. We'll use 3 counting qubits, and 4 target qubits (7 total).
nq = 7 
clb = 0
initial_state = ''
runner_mode = 'shot'
noise_model = ''
shots = 1024

a = 7
N = 15

algorithm_instructions = [
    # 1. Apply Hadamard gates to all 3 counting qubits (qubits 0, 1, 2)
    {"name": "H", "qubits": [0]},
    {"name": "H", "qubits": [1]},
    {"name": "H", "qubits": [2]},
    
    # 2. Make the target register (qubits 3, 4, 5, 6) hold the value "1"
    # Qiskit uses little-endian, so we apply X to the lowest qubit of the target register (qubit 3).
    {"name": "X", "qubits": [3]},
    
    # 3. Apply the custom Modular Exponentiation
    # We pass all 7 qubits. The logic sees 7-4 = 3 control qubits and 4 target qubits.
    {"name": "MOD_EXP", "qubits": [0,1,2,3,4,5,6], "params": [a, N]},
    
    # 4. Apply Inverse Quantum Fourier Transform to the 3 counting qubits
    {"name": "IQFT", "qubits": [0,1,2]},
]

dicccionary = dict_builder(nq, clb, initial_state, algorithm_instructions, runner_mode, noise_model, shots)


In [4]:
results = simulate_circuit(dicccionary, time.time())

counts = results.get('output').get('results')

# Let's inspect the bitstrings. The first 3 bits are the counting register (the ones we care about)
plot_histogram(counts)
print(counts)

{'0001010': 56, '1101010': 67, '0100000': 60, '0100100': 63, '0100010': 69, '0100110': 80, '0111110': 58, '0001110': 68, '0111010': 66, '1101100': 55, '0111000': 64, '0111100': 57, '0001000': 72, '1101110': 67, '0001100': 54, '1101000': 68}


In [5]:
import math
from fractions import Fraction

def post_process_shor(counts, a, N, num_counting_qubits):
    # 1. Marginalize the counts to only care about the counting register
    counting_counts = {}
    for bitstring, count in counts.items():
        # Our bitstrings were reversed to display as q0..q6. 
        # But Qiskit's IQFT uses the highest index of the block (q2) as the MSB.
        # So we take the first 3 bits and reverse them to properly evaluate the integer
        counting_bits = bitstring[:num_counting_qubits][::-1] 
        counting_counts[counting_bits] = counting_counts.get(counting_bits, 0) + count
        
    # Sort them by most frequently measured
    sorted_counts = sorted(counting_counts.items(), key=lambda x: x[1], reverse=True)
    
    print("Most frequent measurements (Counting Register):")
    factors = set()
    for bitstring, count in sorted_counts[:4]: # look at the top 4 peaks
        measured_int = int(bitstring, 2)
        if measured_int == 0:
            continue # Phase 0 yields no information
            
        phase = measured_int / (2**num_counting_qubits)
        
        # 2. Continued fractions to find the period 'r'
        frac = Fraction(phase).limit_denominator(N)
        r = frac.denominator
        
        print(f" - Measured: {bitstring} (int {measured_int}) -> Phase: {phase} -> Guessed r: {r}")
        
        # 3. Factor calculation
        if r % 2 == 0:
            guess1 = math.gcd(a**(r//2) - 1, N)
            guess2 = math.gcd(a**(r//2) + 1, N)
            
            if guess1 not in [1, N]: factors.add(guess1)
            if guess2 not in [1, N]: factors.add(guess2)
            
    return factors

factors = post_process_shor(counts, 7, 15, 3)
print(f"\nCalculated Factors for 15: {list(factors)}")

Most frequent measurements (Counting Register):
 - Measured: 010 (int 2) -> Phase: 0.25 -> Guessed r: 4
 - Measured: 011 (int 3) -> Phase: 0.375 -> Guessed r: 8
 - Measured: 110 (int 6) -> Phase: 0.75 -> Guessed r: 4

Calculated Factors for 15: [3, 5]
